In [1]:
from transformers import logging
logging.set_verbosity_error()  # mute notices


In [2]:
import pandas as pd
from transformers import pipeline
import json

data = pd.read_csv("../data/example_dataset.csv")
textlist = list(data["text"])


In [3]:
#| label: classify
generate = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")

results = []
for text in textlist:
    instructions = [
        {"role": "system",
         "content": "You are a trained assistant for content analysis who determines whether a text expresses fear. Always answer precisely in JSON format with fear (true or false) and reasoning (a short justification in English). Return only the JSON response, starting with '{' and ending with '}', with these two parameters"},
        {"role": "user", "content": text}]
    outputs = generate(instructions, max_new_tokens=256, do_sample=False)
    results.append({"text": text, "response": outputs[0]["generated_text"][-1]['content']})

def parse_response(response):
    response = response.strip().strip("`").removeprefix("json").strip()
    try:
        parsed = json.loads(response)
        return parsed['fear'], parsed['reasoning']
    except (json.JSONDecodeError, KeyError):
        return None, None

parsed_data = [(entry['text'], *parse_response(entry['response'])) for entry in results]
parsed_data = pd.DataFrame(parsed_data, columns=['text', 'fear', 'reasoning'])
print(parsed_data)

                                                text   fear  \
0       I feel helpless and afraid like never before   True   
1  No commute; sunny day; my kids laughing; being...  False   
2  Feeling scared and lonely, can't wait for this...   True   
3  It's gonna get tougher but we will get through...  False   
4  I am terrified by what is happening. I’m scare...   True   
5  Stay inside, stay safe, and this madness will ...  False   
6  I am scared that I will get sick and die, and ...   True   
7  Feeling relaxed and happy but I understand tha...  False   
8  I am very scared and worried about contracting...   True   
9     I'm confident we will get through this crisis.  False   

                                           reasoning  
0  The statement expresses feelings of helplessne...  
1  The text describes positive experiences such a...  
2  The text expresses feelings of being scared an...  
3  The statement suggests perseverance and determ...  
4  The text expresses clear con